In [3]:
from pathlib import Path
import polars as pl
import tiktoken

%load_ext autoreload
%autoreload 2
import sys

sys.path.append(
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/chroma_research_repos/generative-benchmarking"
)

import json
import os
from datetime import datetime
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
import polars as pl
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from functions.visualize import *
from functions.chroma import *
from functions.embed import *
from functions.evaluate import *
from functions.llm import *
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI as OpenAIClient
from tqdm.auto import tqdm

from context_is_king.rag_pipeline import DATA_DIR

load_dotenv()
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# assert OPENAI_API_KEY is not None, "OPENAI_API_KEY is not set"

ORQ_API_KEY = os.getenv("ORQ_API_KEY")
assert ORQ_API_KEY is not None, "ORQ_API_KEY is not set"

chroma_client = chromadb.PersistentClient(path="../chroma_wikitext")
openai_client = OpenAIClient(api_key=ORQ_API_KEY, base_url="https://api.orq.ai/v2/proxy")

DATA_DIR = Path("/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/") / "data"
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

encoding = tiktoken.encoding_for_model("gpt-4o")


2025-09-14 21:01:02.422 | DEBUG    | context_is_king.rag_pipeline:<module>:32 - DATA_DIR=PosixPath('/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/src/data')


In [4]:
chroma_client.list_collections()

[Collection(name=longmemeval_conversations)]

In [7]:
client = chromadb.PersistentClient(path="../data/chroma_db")
client.list_collections()

[Collection(name=wikitext_nq_question_answer)]

In [6]:
collection = client.get_collection("wikitext_nq_question_answer")

In [13]:
from openai import OpenAI
openai_client = OpenAI(
      api_key=os.getenv("ORQ_API_KEY"),
      base_url="https://api.orq.ai/v2/proxy"
  )
def get_768_embedding(text: str):
      response = openai_client.embeddings.create(
          model="azure/text-embedding-3-small",
          input=text,
          dimensions=768  # Match collection expectation
      )
      return response.data[0].embedding

query_embedding = get_768_embedding("your query")

In [9]:
print(collection.count())
results = collection.query(query_embeddings=query_embedding, n_results=5, where={"quality_filtered": True})

NameError: name 'collection' is not defined

In [15]:
results

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [ ]:
from dotenv import load_dotenv
import os

# Load environment properly
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('ORQ_API_KEY', '')

from context_is_king.models import ModelInterface

# Test with a working GPT model
interface = ModelInterface()
await interface.query_model_async(
    model_name='gemini-2.5-flash',
    prompt='test',
    system_prompt='',
    temperature=0.1,
    max_tokens=100,
    timeout=30,
    max_retries=1,
    base_delay=0,
)

🔗 Initialized Model Interface
📡 API Base URL: https://api.orq.ai/v2/proxy
🤖 Available Models: 13
🔄 Rate Limiting: 5 retries, 1.0s base delay


QueryResult(model_name='gemini-2.5-flash', prompt='test', response='Hello! How can I help you today?', success=True, error_message=None, response_time=0.7588567733764648, prompt_tokens=1, completion_tokens=9, total_tokens=10, estimated_cost=5e-05, temperature=0.1, max_tokens=100, context_size=1, timestamp=1757071541.936629)

In [ ]:
# client.delete_collection("longmemeval_conversations")

In [ ]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

In [2]:
import polars as pl

df = pl.scan_ndjson("~/Downloads/v1.0 simplified nq dev all.jsonl")
df.head(2).collect()

annotations,document_html,document_title,document_tokens,document_url,example_id,long_answer_candidates,question_text,question_tokens
list[struct[4]],str,str,list[struct[4]],str,i64,list[struct[5]],str,list[str]
"[{null,{92,67824,925,66429,808},[{66817,837,66588,816}],""NONE""}, {6237931520544082939,{92,67824,925,66429,808},[{66609,819,66588,816}],""NONE""}, … {5015853435362506856,{92,67824,925,66429,808},[{66609,819,66595,817}],""NONE""}]","""<!DOCTYPE html> <HTML class=""c…","""Therefore sign""","[{101,false,92,""Therefore""}, {106,false,102,""sign""}, … {103618,true,103613,""</Ul>""}]","""https://en.wikipedia.org//w/in…",6915606477668963399,"[{66428,808,41427,14,true}, {41683,20,41505,15,false}, … {75257,1540,74711,1481,true}]","""what do the 3 dots mean in mat…","[""what"", ""do"", … ""math""]"
"[{null,{-1,-1,-1,-1,-1},[],""NONE""}, {5211794810959200573,{-1,-1,-1,-1,-1},[],""NONE""}, … {null,{-1,-1,-1,-1,-1},[],""NONE""}]","""<!DOCTYPE html> <HTML class=""c…","""Watchman (law enforcement)""","[{100,false,92,""Watchman""}, {102,false,101,""(""}, … {130545,true,130540,""</Ul>""}]","""https://en.wikipedia.org//w/in…",-4505971823174084926,"[{43267,115,41595,30,true}, {42120,52,41673,32,false}, … {74963,3536,74701,3518,true}]","""when was the writ watch invent…","[""when"", ""was"", … ""who""]"


In [3]:
token_counts = (
    df.with_columns(
        pl.col("document_html")
        .map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int64)
        .alias("document_tokens")
    )
    .select("document_tokens")
    .collect()
)
token_counts

document_tokens
i64
34241
43051
82144
59866
46280
…
58626
83301
78043


In [ ]:
token_counts.select(
    pl.col("document_tokens").min().alias("min"),
    pl.col("document_tokens").max().alias("max"),
    pl.col("document_tokens").mean().alias("mean"),
)

min,max,mean
i64,i64,f64
15146,548918,76573.488378


In [ ]:
import altair as alt

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [ ]:
hist_values = (
    token_counts["document_tokens"]
    .hist(bins=[0, 50000, 100000, 150000, 200000, 250000, 300000, 350000, 400000, 450000, 500000])
    .with_columns(pl.col("category").cast(pl.Utf8))
)
hist_values

breakpoint,category,count
f64,str,u32
50000.0,"""[0.0, 50000.0]""",3050
100000.0,"""(50000.0, 100000.0]""",3024
150000.0,"""(100000.0, 150000.0]""",1081
200000.0,"""(150000.0, 200000.0]""",381
250000.0,"""(200000.0, 250000.0]""",156
300000.0,"""(250000.0, 300000.0]""",81
350000.0,"""(300000.0, 350000.0]""",37
400000.0,"""(350000.0, 400000.0]""",7
450000.0,"""(400000.0, 450000.0]""",1


In [24]:
hist_values.plot.bar(y="count", x=alt.X("category", sort=None))

alt.Chart(...)

In [19]:
len([x for x in samples[0]["long_answer_candidates"] if x["top_level"] == True])

14

In [ ]:
samples = df.head().collect().to_pandas().to_dict(orient="index")
samples

In [15]:
len(samples[0]["document_html"])

103968

In [ ]:
tokens = encoding.encode(samples[0]["document_html"])

len(tokens)

34241

In [ ]:
# Alternative simpler approach for mode calculation
from collections import Counter


def get_mode_tokens(annotations):
    """Get mode of start and end tokens from annotations, excluding -1 values"""
    start_tokens = []
    end_tokens = []

    for annotation in annotations:
        if "long_answer" in annotation:
            start_token = annotation["long_answer"].get("start_token", -1)
            end_token = annotation["long_answer"].get("end_token", -1)

            if start_token != -1:
                start_tokens.append(start_token)
            if end_token != -1:
                end_tokens.append(end_token)

    if not start_tokens or not end_tokens:
        return [-1, -1]

    # Get mode (most common value) or None if no valid tokens
    mode_start = Counter(start_tokens).most_common(1)[0][0] if start_tokens else -1
    mode_end = Counter(end_tokens).most_common(1)[0][0] if end_tokens else -1

    return mode_start, mode_end


# Apply to DataFrame using map_elements (for complex operations)
df_with_simple_mode = (
    df.with_columns(
        [
            pl.col("annotations")
            .map_elements(lambda x: get_mode_tokens(x), return_dtype=pl.List(pl.Int64))
            .alias("mode_tokens")
        ]
    )
    .with_columns(
        [
            pl.col("mode_tokens").list.first().alias("mode_start_token"),
            pl.col("mode_tokens").list.last().alias("mode_end_token"),
        ]
    )
    # .with_columns(
    #     [
    #         pl.col("mode_start_token").map_elements(lambda x: print(f"start: {x}, type: {type(x)}") or x),
    #         pl.col("mode_end_token").map_elements(lambda x: print(f"end: {x}, type: {type(x)}") or x),
    #         pl.col("document_tokens").map_elements(lambda x: print(f"len: {len(x)}, {x[:3]}")),
    #     ]
    # )
    .with_columns(
        [
            # Create sliced tokens using mode values
            pl.when((pl.col("mode_start_token") != -1) & (pl.col("mode_end_token") != -1))
            .then(
                pl.col("document_tokens").list.slice(
                    pl.col("mode_start_token"), pl.col("mode_end_token") - pl.col("mode_start_token")
                )
            )
            .otherwise(None)
            .alias("sliced_document_tokens_simple_mode")
        ]
    )
    .with_columns(
        pl.col("sliced_document_tokens_simple_mode")
        .list.eval(pl.element().struct.field("token"))
        .list.join(" ")
        .alias("long_answer_text")
    )
)

print("DataFrame with simple mode calculation:")
df_with_simple_mode.select("document_url", "question_text", "long_answer_text", "document_html").sink_parquet(
    DATA_DIR / "processed/nq_question_answer.parquet"
)

DataFrame with simple mode calculation:


In [55]:
df_with_simple_mode.select("document_url", "question_text", "long_answer_text", "document_html").collect()

KeyboardInterrupt: 

## Create Embeddings

In [32]:
wiki_qa = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer.parquet")
print(wiki_qa.select(pl.len()).collect())
wiki_qa.head().collect()

shape: (1, 1)
┌──────┐
│ len  │
│ ---  │
│ u32  │
╞══════╡
│ 7830 │
└──────┘


document_url,question_text,long_answer_text,document_html
str,str,str,str
"""https://en.wikipedia.org//w/in…","""what do the 3 dots mean in mat…","""<P> In logical argument and ma…","""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""when was the writ watch invent…",null,"""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""who wrote the song photograph …","""<P> `` Photograph '' is a song…","""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""who is playing the halftime sh…","""<P> The Super Bowl 50 Halftime…","""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""star wars the clone wars anaki…","""<P> Matthew MacKendree `` Matt…","""<!DOCTYPE html> <HTML class=""c…"


In [47]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1900, chunk_overlap=200)
wiki_qa.with_columns(
    pl.col("document_html")
    .map_elements(lambda x: BeautifulSoup(x, "html.parser").get_text(), return_dtype=pl.String)
    .map_elements(lambda x: text_splitter.split_text(x), return_dtype=pl.List(pl.String))
    .alias("chunked_prompt")
).sink_parquet(DATA_DIR / "processed/nq_question_answer_chunked.parquet")

In [48]:
wiki_chunked = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked.parquet")

In [49]:
from uuid import uuid4
import re


def generate_ids() -> str:
    return str(uuid4())


documents = (
    pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked.parquet")
    # .limit(50)
    .select("chunked_prompt", "document_url")
    .explode("chunked_prompt")
    .with_columns(
        pl.col("chunked_prompt").str.len_chars().alias("context_length"),
        pl.col("chunked_prompt")
        .map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int64)
        .alias("token_count"),
        pl.col("chunked_prompt").map_elements(lambda x: re.sub(r"\n{3,}", "\n\n", x), return_dtype=pl.String()),
        pl.col("chunked_prompt").map_elements(lambda _: generate_ids(), return_dtype=pl.String()).alias("unique_id"),
    )
).collect()
# documents = documents.with_columns(pl.Series(generate_ids(documents.height)).alias("unique_id"))
print(documents.shape)
print(documents.select(pl.col("context_length").mean()))
print(documents.select(pl.col("token_count").mean().alias("avg_tokens")))
documents.head()

fn_target = DATA_DIR / "processed/nq_question_answer_chunked_exploded.delta"

documents.write_delta(fn_target, mode="overwrite")

(212044, 5)
shape: (1, 1)
┌────────────────┐
│ context_length │
│ ---            │
│ f64            │
╞════════════════╡
│ 1382.830153    │
└────────────────┘
shape: (1, 1)
┌────────────┐
│ avg_tokens │
│ ---        │
│ f64        │
╞════════════╡
│ 355.43375  │
└────────────┘


In [3]:
documents = pl.read_delta(DATA_DIR / "processed/nq_question_answer_chunked_exploded.delta")

In [8]:
corpus_ids = documents.select("unique_id").to_numpy().reshape(-1).tolist()
corpus_documents = documents.select("chunked_prompt").to_numpy().reshape(-1).tolist()
# metadatas = documents.select("document_url").to_dicts()
# print(type(corpus_ids), type(corpus_documents), type(metadatas))

batch_size = 50_000
for batch in range(0, len(corpus_documents), batch_size):
    selected_documents = documents.filter(pl.col("unique_id").is_in(corpus_ids[batch : batch + batch_size]))
    corpus_embeddings = openai_embed_in_batches(
        openai_client=openai_client,
        model="azure/text-embedding-3-small",
        texts=selected_documents.select("chunked_prompt").to_numpy().reshape(-1).tolist(),
        batch_size=100,
    )
    selected_documents = selected_documents.with_columns(pl.Series(corpus_embeddings).alias("embedding")).write_delta(
        DATA_DIR / f"processed/nq_question_answer_chunked_embeddings.delta", mode="append"
    )

Processing OpenAI batches: 100%|██████████| 121/121 [04:08<00:00,  2.06s/it]


In [23]:
lazy_chunks = pl.scan_delta(DATA_DIR / f"processed/nq_question_answer_chunked_embeddings.delta")

In [24]:
lazy_chunks.with_columns(
    pl.col('document_url')
    .map_elements(lambda x: extract_title_from_path(x), return_dtype=pl.String())
    .alias('collection_suffix')
).sink_parquet(DATA_DIR / "processed/nq_question_answer_chunked_embeddings_with_suffix.parquet")

In [4]:
pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked_embeddings_with_suffix.parquet").collect_schema()

Schema([('chunked_prompt', String),
        ('document_url', String),
        ('context_length', Int32),
        ('token_count', Int64),
        ('unique_id', String),
        ('embedding', List(Float64)),
        ('collection_suffix', String)])

In [5]:
import re

def extract_title_from_path(url):
    pattern = r"title=([^&]+)"
    match = re.search(pattern, url)
    return (
        match.group(1).translate({ord(c): None for c in "!@#$():%,^&!/+=."}).strip("!@#$():%,^&!/+=._")
        if match
        else None
    )


# strings = documents.select("document_url").unique().to_numpy().reshape(-1).tolist()

# for doc in strings:
#     print(extract_title_from_path(doc))
#     break

In [10]:
for x in [x for x in chroma_client.list_collections() if "wikiqa" in x.name]:
    chroma_client.delete_collection(x.name)

In [12]:
done = set()

In [8]:
import polars as pl
allowed_df = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked_embeddings_with_suffix.parquet")
approved_url = allowed_df.group_by('document_url').len().filter(pl.col('len') > 100).collect()['document_url'].to_list()

In [17]:
import gc
collections = [x.name for x in chroma_client.list_collections()]
# for file in Path(DATA_DIR / "processed").glob("nq_question_answer_chunked_embeddings_*.parquet"):
nq_with_embeddings = pl.read_delta(DATA_DIR / "processed/nq_question_answer_chunked_embeddings.delta")
for i, (name, chunk) in tqdm(enumerate(nq_with_embeddings.group_by('document_url')), total=nq_with_embeddings.select(pl.col('document_url').n_unique()).collect().item()):
    doc_url = name[0]

    # for id in tqdm(chunk.select("document_url").unique().to_numpy().reshape(-1).tolist()):
    if doc_url not in approved_url:
        continue
    print('Processing ', doc_url)
    collection_name = f"wikiqa_{extract_title_from_path(doc_url)}"
    cache_name = f"{collection_name}_{i}"
    # logger.info(collection_name, collection_name in collections)
    if collection_name in collections:
        chroma_client.delete_collection(collection_name)

    # logger.info(f"Processing {collection_name}")
    corpus_collection = chroma_client.get_or_create_collection(
        name=collection_name, metadata={"hnsw:space": "cosine"}
    )
    if collection_name in collections:
        existing_ids = corpus_collection.get()['ids']
    selected_docs = chunk.filter(pl.col("document_url") == doc_url).filter(~pl.col("unique_id").is_in(existing_ids))
    corpus_ids = selected_docs.select("unique_id").to_numpy().reshape(-1).tolist()
    corpus_documents = selected_docs.select("chunked_prompt").to_numpy().reshape(-1).tolist()
    metadatas = [{**x, 'cache_name': cache_name, 'collection_name': collection_name} for x in selected_docs.select("document_url").to_dicts()]
    embeddings = selected_docs.select("embedding").to_numpy().reshape(-1).tolist()

    # print(len(corpus_ids), len(corpus_documents), len(metadatas), len(selected_docs))

    collection_add_in_batches(
        collection=corpus_collection,
        ids=corpus_ids,
        texts=corpus_documents,
        embeddings=embeddings,
        metadatas=metadatas,
        batch_size=500,
    )
    done.add(f"{collection_name}_{i}")
    del embeddings, corpus_documents, corpus_ids, metadatas
    gc.collect()



Processing  https://en.wikipedia.org//w/index.php?title=History_of_the_United_States&amp;oldid=822282210
Processing  https://en.wikipedia.org//w/index.php?title=List_of_performances_on_Top_of_the_Pops&amp;oldid=818121506
Processing  https://en.wikipedia.org//w/index.php?title=Christmas&amp;oldid=822099419
Processing  https://en.wikipedia.org//w/index.php?title=Solar_eclipse_of_August_21,_2017&amp;oldid=807052172
Processing  https://en.wikipedia.org//w/index.php?title=Human_evolution&amp;oldid=821005897
Processing  https://en.wikipedia.org//w/index.php?title=RMS_Titanic&amp;oldid=805999140
Processing  https://en.wikipedia.org//w/index.php?title=I_Can_Only_Imagine_(MercyMe_song)&amp;oldid=837313786
Processing  https://en.wikipedia.org//w/index.php?title=The_Curse_of_Oak_Island&amp;oldid=832882458
Processing  https://en.wikipedia.org//w/index.php?title=Star_Trek:_Discovery&amp;oldid=838430258
Processing  https://en.wikipedia.org//w/index.php?title=Cell_nucleus&amp;oldid=811624751
Processi

In [17]:
selected_docs.select("document_url").to_dicts()

[{'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of_India&amp;oldid=838501027'},
 {'document_url': 'https://en.wikipedia.org//w/index.php?title=Economy_of

In [11]:
len(chroma_client.list_collections())

3752

In [12]:
import deltalake
from deltalake.table import TableOptimizer

non_similar_samples = []

for collection_name in tqdm([x for x in chroma_client.list_collections() if "wikiqa" in x.name][:100]):
    corpus_collection = chroma_client.get_collection(name=collection_name.name)
    collection_samples = []
    corpus = get_collection_items(collection=corpus_collection)
    vals = [
        {
            "collection": collection_name.name,
            "id": id,
            "document": item["document"],
            "embedding": item["embedding"].tolist(),
        }
        for id, item in corpus.items()
    ]
    df = pl.DataFrame(
        vals, schema={"collection": pl.Utf8, "id": pl.Utf8, "document": pl.Utf8, "embedding": pl.List(pl.Float64)}
    )
    samples = df.sample(n=min(25, len(df)))
    for sample in samples.iter_rows(named=True):
        results = corpus_collection.query(query_embeddings=sample["embedding"], n_results=2)
        if results["distances"][0][1] > 0.4:
            collection_samples.append(sample)
    # logger.debug(f"Found {len(collection_samples)} non-similar samples for {collection_name.name}")
    logger.debug(f"Processing {collection_name.name}. Adding {len(collection_samples)} non-similar samples")
    # non_similar_samples.extend(collection_samples)
    # non_similar_samples_df = pl.DataFrame(non_similar_samples)
    if collection_samples:
        pl.DataFrame(collection_samples).write_delta(DATA_DIR / "processed/non_similar_samples.delta", mode="append")

# corpus = {
#     id: {"document": document, "embedding": embedding}
#     for id, document, embedding in zip(corpus_ids, corpus_documents, corpus_embeddings)
# }
# len(corpus_ids), len(corpus_documents), len(corpus_embeddings)
# non_similar_samples_df.write_parquet(DATA_DIR / "processed/non_similar_samples.parquet")

dt = deltalake.DeltaTable(DATA_DIR / "processed/non_similar_samples.delta")
to = TableOptimizer(dt).compact()

  0%|          | 0/3752 [00:00<?, ?it/s]

Processing batches: 100%|██████████| 1/1 [00:00<00:00, 110.02it/s]
2025-08-30 13:06:26.827 | DEBUG    | __main__:<module>:28 - Processing wikiqa_Centurion_Card. Adding 1 non-similar samples
Processing batches: 100%|██████████| 1/1 [00:00<00:00, 38.60it/s]
2025-08-30 13:06:30.801 | DEBUG    | __main__:<module>:28 - Processing wikiqa_Public_Procurement_and_Disposal_of_Public_Assets_Authority. Adding 1 non-similar samples
Processing batches: 100%|██████████| 1/1 [00:00<00:00, 105.02it/s]
2025-08-30 13:06:34.167 | DEBUG    | __main__:<module>:28 - Processing wikiqa_Chinese_garden. Adding 2 non-similar samples
Processing batches: 100%|██████████| 1/1 [00:00<00:00, 196.46it/s]
2025-08-30 13:06:37.444 | DEBUG    | __main__:<module>:28 - Processing wikiqa_Pay_Commission. Adding 2 non-similar samples
Processing batches: 100%|██████████| 1/1 [00:00<00:00, 229.66it/s]
2025-08-30 13:06:40.757 | DEBUG    | __main__:<module>:28 - Processing wikiqa_The_Emperor27s_New_Clothes. Adding 2 non-similar sam

KeyboardInterrupt: 

In [13]:
dt = deltalake.DeltaTable(DATA_DIR / "processed/non_similar_samples.delta")
to = TableOptimizer(dt).compact()

In [4]:
sampled_delta = pl.scan_delta(DATA_DIR / "processed/non_similar_samples.delta").unique("id").collect().sample(1000)
sampled_delta.select(pl.len())

len
u32
1000


In [5]:
context = "We are find good RAG/LLM benchmark data, compared to long context. For this, we are interested in data that contains queryable information. If things are just chit-chat, we can drop it. The purpose is to create questions and answers that have one place in the text where they can be found."
example_queries = """
    How many movies did i watch recently?
    What is a good way to declutter my bookshelf?
    How large was the aquarium that i recently set up? 
    """
relevance = f"The document is relevant to the following context: {context}"
completeness = """The document is complete, meaning that it contains useful information to answer queries and does not 
only serve as an introduction to the main content that users may be looking for. It should also succeed if there is
self-contained information in the statement, about which we can ask factual questions."""
uniqueness = """The document is unique, meaning that it does not contain information that is already present in other documents."""

criteria = [relevance, completeness, uniqueness]
criteria_labels = ["relevance", "completeness", "uniqueness"]

### 3.2 Filter Documents

We filter our documents using gpt-4o-mini. Batching functions are also available in `llm.py`.

In [9]:
sampled_delta.head()

collection,id,document,embedding
str,str,str,list[f64]
"""wikiqa_Lost_Dutchman27s_Gold_M…","""ee685ef6-ca5d-4989-ac83-706035…","""Privacy policy About Wikipedia…","[0.052456, -0.017897, … -0.047907]"
"""wikiqa_Wurtz_reaction""","""78c80406-5cfc-43f9-bcfe-c55cb7…","""Examples and reaction conditio…","[0.012913, -0.006684, … 0.024102]"
"""wikiqa_Get_the_Party_Started""","""a9caa0cb-7262-4b94-aa81-b03c75…","""enthusiastic R&B of the origin…","[-0.002424, -0.009661, … -0.007904]"
"""wikiqa_All_These_Things_That_I…","""ea58a5eb-c158-467b-9e78-461a5a…","""Concert tours Hot Fuss Tour…","[-0.033472, -0.053955, … -0.031365]"
"""wikiqa_Climate_of_South_Africa""","""87336797-5260-4a5e-81aa-33e73b…","""in May 1956, August 1962, June…","[0.001818, 0.017883, … -0.013166]"


In [10]:
corpus_ids = sampled_delta.select("id").to_numpy().reshape(-1).tolist()
corpus_documents = sampled_delta.select("document").to_numpy().reshape(-1).tolist()
metadatas = sampled_delta.select("collection").to_dicts()

In [11]:
from functions.llm import (
    filter_documents_nvidia,
    filter_documents_async,
    filter_documents_parallel,
    create_filter_documents_batch,
)
from openai import AsyncOpenAI, AzureOpenAI

api_key = os.getenv("ORQ_API_KEY")
base_url = "https://api.orq.ai/v2/proxy"
async_openai_client = AsyncOpenAI(api_key=api_key, base_url=base_url)

In [18]:
len(set(corpus_ids)), len(corpus_ids)

(1000, 1000)

In [19]:
duplicates = pl.Series(corpus_ids).value_counts().filter(pl.col("count") > 1)
duplicates

,count
str,u32


In [ ]:
id = create_filter_documents_batch(
    client=openai_client,
    model="azure/gpt-5-mini",
    documents=corpus_documents,
    ids=corpus_ids,
    criteria=criteria,
    criteria_labels=criteria_labels,
    max_requests_per_batch=1000,
)
id

NotFoundError: 404 Not Found

: 

In [12]:
filtered_document_ids = await filter_documents_parallel(
    client=async_openai_client,
    model="azure/gpt-5-nano",
    documents=corpus_documents,
    ids=corpus_ids,
    criteria=criteria,
    criteria_labels=criteria_labels,
    tokens_per_minute=300000,
)
filtered_document_ids

2025-08-31 12:43:06.207 | INFO     | functions.llm:filter_documents_async:283 - Starting 3000 async evaluations with max 50 concurrent requests


Processing evaluations:   0%|          | 0/3000 [00:00<?, ?it/s]

2025-08-31 12:50:26.727 | INFO     | functions.llm:filter_documents_async:334 - Filtered 6 documents out of 1000 total
2025-08-31 12:50:26.730 | INFO     | functions.llm:filter_documents_parallel:382 - filter_documents_parallel returning 2 filtered IDs


(['2004ff80-60d8-4b33-9903-09cdc93c31b3',
  '29e0e032-8dc6-4fd8-845f-f5e2b5865680',
  'a77dbe29-6235-4c9d-803d-1df9c78e63a4',
  '57247594-a236-4539-b7a6-666aacbc5eb1',
  '31971364-1caf-4a6b-aecd-aedc5923b480',
  'a6d5f1e5-7c8a-40b2-9276-0c4084a0dd19'],
 {'6c86ab5f-855d-463d-845d-b28c89df0fce': {'uniqueness': False,
   'relevance': True,
   'completeness': True},
  '4710f8aa-dccf-4388-aeff-76f5074a85ea': {'completeness': False,
   'relevance': False,
   'uniqueness': False},
  'a27d736e-c72b-4b1a-a4e2-03e2b90e206f': {'relevance': False,
   'completeness': False,
   'uniqueness': False},
  '06e62155-dc3a-4f7a-bc7b-4602abc04f96': {'completeness': False,
   'uniqueness': False,
   'relevance': False},
  '29876a01-8d64-4dde-983d-71b6fe368d53': {'completeness': False,
   'uniqueness': False,
   'relevance': True},
  'd3be71a6-6568-4a01-b402-7d63bf3d86f1': {'uniqueness': False,
   'completeness': False,
   'relevance': False},
  '99e462c2-18a6-4225-ab2c-828df0bc278f': {'completeness': True,
 

In [19]:
filtered_document_ids[0]

['bd61c0c1-1f28-4b07-b67f-2b555db9ab57',
 '5bd073a6-3973-414d-b9ef-5f8f36ce6c10']

In [13]:
prepped_data = []
for k, v in filtered_document_ids[1].items():
    prepped_data.append({"id": k, **v})

prepped_data[:2]

[{'id': '6c86ab5f-855d-463d-845d-b28c89df0fce',
  'uniqueness': False,
  'relevance': True,
  'completeness': True},
 {'id': '4710f8aa-dccf-4388-aeff-76f5074a85ea',
  'completeness': False,
  'relevance': False,
  'uniqueness': False}]

In [14]:
chatgpt_filters_df = pl.DataFrame(prepped_data)
chatgpt_filters_df

id,uniqueness,relevance,completeness
str,bool,bool,bool
"""6c86ab5f-855d-463d-845d-b28c89…",false,true,true
"""4710f8aa-dccf-4388-aeff-76f507…",false,false,false
"""a27d736e-c72b-4b1a-a4e2-03e2b9…",false,false,false
"""06e62155-dc3a-4f7a-bc7b-4602ab…",false,false,false
"""29876a01-8d64-4dde-983d-71b6fe…",false,true,false
…,…,…,…
"""57cca4d6-7847-486c-8d28-c6c2d2…",false,false,false
"""255ee098-c9e9-4166-954f-55e754…",false,true,true
"""b12b64f4-6ba4-402c-9858-e18070…",false,true,true


In [15]:
chatgpt_filters_df.join(sampled_delta, on="id", how="left").with_columns(
    pl.col("document").str.len_chars().alias("document_length")
).filter(pl.col("document_length") > 100)

id,uniqueness,relevance,completeness,collection,document,embedding,document_length
str,bool,bool,bool,str,str,list[f64],u32
"""6c86ab5f-855d-463d-845d-b28c89…",false,true,true,"""wikiqa_Pre-Columbian_era""","""Numerous pre-Columbian societi…","[0.028962, -0.02533, … -0.026098]",381
"""4710f8aa-dccf-4388-aeff-76f507…",false,false,false,"""wikiqa_The_Vampire_Diaries""","""Personal tools Not logged inT…","[0.00511, 0.026274, … -0.049983]",1499
"""a27d736e-c72b-4b1a-a4e2-03e2b9…",false,false,false,"""wikiqa_Poverty_threshold""","""Tools What links hereRelated …","[-0.028022, 0.017003, … -0.050899]",962
"""29876a01-8d64-4dde-983d-71b6fe…",false,true,false,"""wikiqa_United_Methodist_Church…","""Firestone Chapel The …","[0.005641, -0.050796, … -0.013202]",221
"""fa1b9300-13fb-456d-b953-ce2e59…",false,true,false,"""wikiqa_Bias""","""Jump up ^ Gross, Neil (9 April…","[0.005143, 0.042184, … -0.056955]",222
…,…,…,…,…,…,…,…
"""5b3b9096-373b-49d5-9cc2-69094c…",false,false,true,"""wikiqa_Extreme_programming""","""The first time I was asked to …","[0.001277, 0.024318, … -0.025326]",385
"""8c456e55-2fae-4f15-b702-310eea…",false,true,false,"""wikiqa_BoC3B6tes""","""was Qigong, the Seven Dukes, w…","[-0.053566, 0.016452, … -0.013607]",148
"""255ee098-c9e9-4166-954f-55e754…",false,true,true,"""wikiqa_What_Happens_to_My_Fami…","""Episode Broadcast date TNmS ra…","[0.019079, 0.009635, … 0.007554]",1844


In [5]:
corpus_ids[0]

'fcc3e290-c2b2-42a8-ad95-0f5ce05dc366'